# Batch CLIP Detection untuk Dataset Skema 2

Notebook ini membaca `dataset/labels_skema2.xlsx`, mencocokkan kolom `File` dengan gambar di `dataset/images_skema2`, lalu menambahkan top-3 label CLIP berdasarkan kamus 4A/Scene. Output disimpan ke `dataset/clip-detection.csv`.

Output tidak menyimpan probabilitas; CLIP hanya dipakai untuk menentukan urutan label teratas.

## Setup

Jika package belum tersedia, jalankan sekali:

```python
# %pip install -q torch transformers pillow pandas
```

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path
from zipfile import ZipFile
from xml.etree import ElementTree as ET
import re

import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from transformers import CLIPModel, CLIPProcessor


def find_existing_path(*candidates: str) -> Path:
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    raise FileNotFoundError(f"Tidak ada path yang ditemukan dari kandidat: {candidates}")


DATASET_XLSX = find_existing_path("dataset/labels_skema2.xlsx", "../dataset/labels_skema2.xlsx")
IMAGE_DIR = find_existing_path("dataset/images_skema2", "../dataset/images_skema2")
OUTPUT_CSV = DATASET_XLSX.parent / "clip-detection.csv"

MODEL_NAME = "openai/clip-vit-base-patch32"
LOCAL_FILES_ONLY = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TOP_K = 3
BATCH_SIZE = 16

CLIP_COLUMNS = [
    "Clip1_Category", "Clip1_Label",
    "Clip2_Category", "Clip2_Label",
    "Clip3_Category", "Clip3_Label",
]

print(f"Dataset: {DATASET_XLSX}")
print(f"Folder gambar: {IMAGE_DIR}")
print(f"Output CSV: {OUTPUT_CSV}")
print(f"Device: {DEVICE}")

In [ ]:
LABELS = [
    # Attraction
    {"category": "Attraction", "label_en": "beach", "label_id": "pantai"},
    {"category": "Attraction", "label_en": "mountain", "label_id": "gunung"},
    {"category": "Attraction", "label_en": "waterfall", "label_id": "air terjun"},
    {"category": "Attraction", "label_en": "lake", "label_id": "danau"},
    {"category": "Attraction", "label_en": "sea", "label_id": "laut"},
    {"category": "Attraction", "label_en": "island", "label_id": "pulau"},
    {"category": "Attraction", "label_en": "forest", "label_id": "hutan"},
    {"category": "Attraction", "label_en": "sunset", "label_id": "matahari terbenam"},
    {"category": "Attraction", "label_en": "sunrise", "label_id": "matahari terbit"},
    {"category": "Attraction", "label_en": "natural landscape", "label_id": "lanskap alam"},
    {"category": "Attraction", "label_en": "tropical scenery", "label_id": "pemandangan tropis"},
    {"category": "Attraction", "label_en": "volcano", "label_id": "gunung berapi"},
    {"category": "Attraction", "label_en": "cliff", "label_id": "tebing"},
    {"category": "Attraction", "label_en": "river", "label_id": "sungai"},
    {"category": "Attraction", "label_en": "cave", "label_id": "gua"},
    {"category": "Attraction", "label_en": "cultural attraction", "label_id": "daya tarik budaya"},
    {"category": "Attraction", "label_en": "traditional house", "label_id": "rumah tradisional"},
    {"category": "Attraction", "label_en": "temple", "label_id": "candi atau pura"},
    {"category": "Attraction", "label_en": "monument", "label_id": "monumen"},
    {"category": "Attraction", "label_en": "historical building", "label_id": "bangunan bersejarah"},
    {"category": "Attraction", "label_en": "scenic view", "label_id": "pemandangan indah"},
    {"category": "Attraction", "label_en": "recreation area", "label_id": "area rekreasi"},
    {"category": "Attraction", "label_en": "camping area", "label_id": "area berkemah"},
    {"category": "Attraction", "label_en": "crowd of tourists", "label_id": "keramaian wisatawan"},
    {"category": "Attraction", "label_en": "outdoor activity", "label_id": "aktivitas luar ruangan"},
    {"category": "Attraction", "label_en": "snorkeling", "label_id": "snorkeling"},
    {"category": "Attraction", "label_en": "diving", "label_id": "menyelam"},
    {"category": "Attraction", "label_en": "hiking", "label_id": "mendaki"},
    {"category": "Attraction", "label_en": "beach recreation", "label_id": "rekreasi pantai"},

    # Accessibility
    {"category": "Accessibility", "label_en": "road", "label_id": "jalan"},
    {"category": "Accessibility", "label_en": "highway", "label_id": "jalan raya"},
    {"category": "Accessibility", "label_en": "bridge", "label_id": "jembatan"},
    {"category": "Accessibility", "label_en": "harbor", "label_id": "pelabuhan"},
    {"category": "Accessibility", "label_en": "port", "label_id": "pelabuhan"},
    {"category": "Accessibility", "label_en": "airport", "label_id": "bandara"},
    {"category": "Accessibility", "label_en": "parking area", "label_id": "area parkir"},
    {"category": "Accessibility", "label_en": "transportation", "label_id": "transportasi"},
    {"category": "Accessibility", "label_en": "vehicle", "label_id": "kendaraan"},
    {"category": "Accessibility", "label_en": "motorcycle", "label_id": "motor"},
    {"category": "Accessibility", "label_en": "car", "label_id": "mobil"},
    {"category": "Accessibility", "label_en": "bus", "label_id": "bus"},
    {"category": "Accessibility", "label_en": "boat", "label_id": "perahu"},
    {"category": "Accessibility", "label_en": "ferry", "label_id": "kapal feri"},
    {"category": "Accessibility", "label_en": "traffic sign", "label_id": "rambu lalu lintas"},
    {"category": "Accessibility", "label_en": "pedestrian path", "label_id": "jalur pejalan kaki"},
    {"category": "Accessibility", "label_en": "entrance gate", "label_id": "gerbang masuk"},
    {"category": "Accessibility", "label_en": "tourism access road", "label_id": "jalan akses wisata"},
    {"category": "Accessibility", "label_en": "public transportation", "label_id": "transportasi umum"},
    {"category": "Accessibility", "label_en": "dock", "label_id": "dermaga"},
    {"category": "Accessibility", "label_en": "pathway", "label_id": "jalur"},
    {"category": "Accessibility", "label_en": "stairs", "label_id": "tangga"},
    {"category": "Accessibility", "label_en": "direction sign", "label_id": "papan petunjuk arah"},

    # Amenities
    {"category": "Amenities", "label_en": "hotel", "label_id": "hotel"},
    {"category": "Amenities", "label_en": "resort", "label_id": "resort"},
    {"category": "Amenities", "label_en": "homestay", "label_id": "homestay"},
    {"category": "Amenities", "label_en": "restaurant", "label_id": "restoran"},
    {"category": "Amenities", "label_en": "cafe", "label_id": "kafe"},
    {"category": "Amenities", "label_en": "food stall", "label_id": "warung makan"},
    {"category": "Amenities", "label_en": "public toilet", "label_id": "toilet umum"},
    {"category": "Amenities", "label_en": "mosque", "label_id": "masjid"},
    {"category": "Amenities", "label_en": "prayer room", "label_id": "mushola"},
    {"category": "Amenities", "label_en": "souvenir shop", "label_id": "toko suvenir"},
    {"category": "Amenities", "label_en": "tourism facility", "label_id": "fasilitas wisata"},
    {"category": "Amenities", "label_en": "parking facility", "label_id": "fasilitas parkir"},
    {"category": "Amenities", "label_en": "gazebo", "label_id": "gazebo"},
    {"category": "Amenities", "label_en": "seating area", "label_id": "area duduk"},
    {"category": "Amenities", "label_en": "swimming pool", "label_id": "kolam renang"},
    {"category": "Amenities", "label_en": "playground", "label_id": "taman bermain"},
    {"category": "Amenities", "label_en": "camping facility", "label_id": "fasilitas berkemah"},
    {"category": "Amenities", "label_en": "market", "label_id": "pasar"},
    {"category": "Amenities", "label_en": "information board", "label_id": "papan informasi"},
    {"category": "Amenities", "label_en": "street lighting", "label_id": "lampu jalan"},
    {"category": "Amenities", "label_en": "wifi area", "label_id": "area wifi"},
    {"category": "Amenities", "label_en": "resting area", "label_id": "area istirahat"},

    # Ancillary
    {"category": "Ancillary", "label_en": "tourism event", "label_id": "acara wisata"},
    {"category": "Ancillary", "label_en": "festival", "label_id": "festival"},
    {"category": "Ancillary", "label_en": "cultural performance", "label_id": "pertunjukan budaya"},
    {"category": "Ancillary", "label_en": "tour guide", "label_id": "pemandu wisata"},
    {"category": "Ancillary", "label_en": "tourism office", "label_id": "kantor pariwisata"},
    {"category": "Ancillary", "label_en": "tourism information center", "label_id": "pusat informasi wisata"},
    {"category": "Ancillary", "label_en": "community activity", "label_id": "aktivitas komunitas"},
    {"category": "Ancillary", "label_en": "local guide", "label_id": "pemandu lokal"},
    {"category": "Ancillary", "label_en": "tourism promotion banner", "label_id": "spanduk promosi wisata"},
    {"category": "Ancillary", "label_en": "tourism organization", "label_id": "organisasi pariwisata"},
    {"category": "Ancillary", "label_en": "security officer", "label_id": "petugas keamanan"},
    {"category": "Ancillary", "label_en": "rescue team", "label_id": "tim penyelamat"},
    {"category": "Ancillary", "label_en": "volunteers", "label_id": "relawan"},
    {"category": "Ancillary", "label_en": "government tourism activity", "label_id": "aktivitas pariwisata pemerintah"},
    {"category": "Ancillary", "label_en": "exhibition", "label_id": "pameran"},
    {"category": "Ancillary", "label_en": "tourism campaign", "label_id": "kampanye pariwisata"},
    {"category": "Ancillary", "label_en": "official tourism signage", "label_id": "papan resmi pariwisata"},
    {"category": "Ancillary", "label_en": "ticket counter", "label_id": "loket tiket"},
    {"category": "Ancillary", "label_en": "tourism management activity", "label_id": "aktivitas pengelolaan wisata"},

    # Scene / Suasana Tambahan
    {"category": "Scene", "label_en": "crowded tourism area", "label_id": "area wisata ramai"},
    {"category": "Scene", "label_en": "quiet place", "label_id": "tempat tenang"},
    {"category": "Scene", "label_en": "relaxing atmosphere", "label_id": "suasana santai"},
    {"category": "Scene", "label_en": "adventure tourism", "label_id": "wisata petualangan"},
    {"category": "Scene", "label_en": "family tourism", "label_id": "wisata keluarga"},
    {"category": "Scene", "label_en": "romantic scenery", "label_id": "pemandangan romantis"},
    {"category": "Scene", "label_en": "urban tourism", "label_id": "wisata perkotaan"},
    {"category": "Scene", "label_en": "rural tourism", "label_id": "wisata pedesaan"},
    {"category": "Scene", "label_en": "eco tourism", "label_id": "ekowisata"},
    {"category": "Scene", "label_en": "marine tourism", "label_id": "wisata bahari"},
    {"category": "Scene", "label_en": "nature tourism", "label_id": "wisata alam"},
    {"category": "Scene", "label_en": "cultural tourism", "label_id": "wisata budaya"},
    {"category": "Scene", "label_en": "night tourism", "label_id": "wisata malam"},
    {"category": "Scene", "label_en": "cloudy sky", "label_id": "langit berawan"},
    {"category": "Scene", "label_en": "sunny weather", "label_id": "cuaca cerah"},
    {"category": "Scene", "label_en": "tropical environment", "label_id": "lingkungan tropis"},
    {"category": "Scene", "label_en": "clean environment", "label_id": "lingkungan bersih"},
    {"category": "Scene", "label_en": "dirty environment", "label_id": "lingkungan kotor"},
    {"category": "Scene", "label_en": "damaged infrastructure", "label_id": "infrastruktur rusak"},
    {"category": "Scene", "label_en": "modern tourism area", "label_id": "area wisata modern"},
    {"category": "Scene", "label_en": "traditional tourism area", "label_id": "area wisata tradisional"},
]

labels_df = pd.DataFrame(LABELS)
print(f"Total label: {len(labels_df)}")
labels_df.groupby("category").size().rename("jumlah_label")

In [ ]:
XLSX_NS = {"a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
COLUMN_RE = re.compile(r"([A-Z]+)")
BUILTIN_DATE_FORMAT_IDS = {14, 15, 16, 17, 22, 27, 30, 36, 45, 46, 47, 50, 57}


def column_ref_to_index(cell_ref: str) -> int:
    match = COLUMN_RE.match(cell_ref)
    if not match:
        raise ValueError(f"Referensi cell tidak valid: {cell_ref}")

    index = 0
    for char in match.group(1):
        index = index * 26 + (ord(char) - ord("A") + 1)
    return index - 1


def excel_serial_to_datetime_text(value: str) -> str:
    try:
        total_seconds = round(float(value) * 86400)
    except (TypeError, ValueError):
        return value

    parsed = datetime(1899, 12, 30) + timedelta(seconds=total_seconds)
    return parsed.strftime("%Y-%m-%d %H:%M:%S")


def load_date_style_ids(zip_file: ZipFile) -> set[str]:
    if "xl/styles.xml" not in zip_file.namelist():
        return set()

    styles = ET.fromstring(zip_file.read("xl/styles.xml"))
    custom_formats = {}
    num_fmts = styles.find("a:numFmts", XLSX_NS)
    if num_fmts is not None:
        for num_fmt in num_fmts.findall("a:numFmt", XLSX_NS):
            custom_formats[int(num_fmt.attrib["numFmtId"])] = num_fmt.attrib.get("formatCode", "").lower()

    date_style_ids = set()
    cell_xfs = styles.find("a:cellXfs", XLSX_NS)
    if cell_xfs is None:
        return date_style_ids

    for style_index, xf in enumerate(cell_xfs.findall("a:xf", XLSX_NS)):
        num_fmt_id = int(xf.attrib.get("numFmtId", 0))
        format_code = custom_formats.get(num_fmt_id, "")
        is_builtin_date = num_fmt_id in BUILTIN_DATE_FORMAT_IDS
        is_custom_date = any(token in format_code for token in ["yy", "dd", "hh"])
        if is_builtin_date or is_custom_date:
            date_style_ids.add(str(style_index))

    return date_style_ids


def load_shared_strings(zip_file: ZipFile) -> list[str]:
    if "xl/sharedStrings.xml" not in zip_file.namelist():
        return []

    shared_root = ET.fromstring(zip_file.read("xl/sharedStrings.xml"))
    shared_strings = []
    for item in shared_root.findall("a:si", XLSX_NS):
        shared_strings.append("".join(text.text or "" for text in item.findall(".//a:t", XLSX_NS)))
    return shared_strings


def read_xlsx_first_sheet(path: Path) -> pd.DataFrame:
    with ZipFile(path) as zip_file:
        shared_strings = load_shared_strings(zip_file)
        date_style_ids = load_date_style_ids(zip_file)
        sheet = ET.fromstring(zip_file.read("xl/worksheets/sheet1.xml"))

        row_maps = []
        max_column_count = 0
        for row in sheet.findall(".//a:sheetData/a:row", XLSX_NS):
            row_values = {}
            for cell in row.findall("a:c", XLSX_NS):
                column_index = column_ref_to_index(cell.attrib["r"])
                max_column_count = max(max_column_count, column_index + 1)
                cell_type = cell.attrib.get("t")

                if cell_type == "inlineStr":
                    value = "".join(text.text or "" for text in cell.findall(".//a:t", XLSX_NS))
                else:
                    value_node = cell.find("a:v", XLSX_NS)
                    if value_node is None:
                        value = ""
                    elif cell_type == "s":
                        value = shared_strings[int(value_node.text)]
                    else:
                        value = value_node.text or ""
                        if cell.attrib.get("s") in date_style_ids:
                            value = excel_serial_to_datetime_text(value)

                row_values[column_index] = value
            row_maps.append(row_values)

        parsed_rows = [
            [row_values.get(index, "") for index in range(max_column_count)]
            for row_values in row_maps
        ]

    headers = parsed_rows[0]
    rows = parsed_rows[1:]
    while headers and headers[-1] == "":
        headers.pop()
        rows = [row[:-1] for row in rows]

    return pd.DataFrame(rows, columns=headers)


df = read_xlsx_first_sheet(DATASET_XLSX)
display(df.head())
print(f"Jumlah baris label: {len(df)}")
print("Kolom:", list(df.columns))

In [ ]:
if "File" not in df.columns:
    raise KeyError("Kolom 'File' tidak ditemukan di labels_skema2.xlsx")
if "Catatan" not in df.columns:
    raise KeyError("Kolom 'Catatan' tidak ditemukan di labels_skema2.xlsx")

file_series = df["File"].astype("string").fillna("").str.strip()
files_from_excel = [file_name for file_name in file_series.tolist() if file_name]
image_paths_by_name = {
    path.name: path
    for path in IMAGE_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
}

missing_images = sorted(set(files_from_excel) - set(image_paths_by_name))
extra_images = sorted(set(image_paths_by_name) - set(files_from_excel))

print(f"Baris dengan File terisi: {len(files_from_excel)}")
print(f"Gambar di folder images_skema2: {len(image_paths_by_name)}")
print(f"File dari Excel yang tidak ada di folder: {len(missing_images)}")
print(f"Gambar di folder yang tidak ada di Excel: {len(extra_images)}")

if missing_images:
    display(pd.DataFrame({"missing_images": missing_images}))
    raise FileNotFoundError("Ada nama file di kolom 'File' yang tidak ditemukan di folder images_skema2.")

if extra_images:
    display(pd.DataFrame({"extra_images": extra_images}))

ordered_image_paths = [image_paths_by_name[file_name] for file_name in files_from_excel]

In [ ]:
try:
    processor = CLIPProcessor.from_pretrained(
        MODEL_NAME,
        local_files_only=LOCAL_FILES_ONLY,
        use_fast=False,
    )
    model = CLIPModel.from_pretrained(
        MODEL_NAME,
        local_files_only=LOCAL_FILES_ONLY,
    ).to(DEVICE)
    model.eval()
except OSError as exc:
    raise RuntimeError(
        "Model CLIP belum ada di cache lokal. Jika ingin download otomatis, ubah "
        "LOCAL_FILES_ONLY = False lalu jalankan ulang cell ini dengan koneksi internet."
    ) from exc

print("Model CLIP siap dipakai.")

In [ ]:
PROMPT_TEMPLATES = [
    "a photo of {label}",
    "a tourism photo showing {label}",
    "a travel destination with {label}",
    "an Instagram travel photo of {label}",
]


def make_prompts(label_en: str) -> list[str]:
    return [template.format(label=label_en) for template in PROMPT_TEMPLATES]


@torch.inference_mode()
def build_label_features(label_items: list[dict]) -> torch.Tensor:
    prompts = [prompt for item in label_items for prompt in make_prompts(item["label_en"])]
    inputs = processor(text=prompts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    text_features = model.get_text_features(**inputs)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    text_features = text_features.reshape(len(label_items), len(PROMPT_TEMPLATES), -1).mean(dim=1)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    return text_features


def load_image(image_path: Path) -> Image.Image:
    return Image.open(image_path).convert("RGB")


@torch.inference_mode()
def predict_clip_topk_batch(image_paths: list[Path], label_items: list[dict], top_k: int = TOP_K, batch_size: int = BATCH_SIZE) -> dict[str, list[dict]]:
    label_features = build_label_features(label_items)
    predictions = {}
    total = len(image_paths)

    for start in range(0, total, batch_size):
        batch_paths = image_paths[start : start + batch_size]
        valid_paths = []
        images = []

        for image_path in batch_paths:
            try:
                images.append(load_image(image_path))
                valid_paths.append(image_path)
            except Exception as exc:
                print(f"Gagal memproses {image_path.name}: {exc}")

        if not images:
            continue

        inputs = processor(images=images, return_tensors="pt").to(DEVICE)
        image_features = model.get_image_features(**inputs)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        similarities = image_features @ label_features.T
        _, top_indices = similarities.topk(top_k, dim=1)

        for image_path, indices in zip(valid_paths, top_indices.cpu().tolist()):
            predictions[image_path.name] = [label_items[index] for index in indices]

        done = min(start + len(batch_paths), total)
        print(f"Progress CLIP: {done}/{total} gambar", end="\r")

    print(f"Progress CLIP: {total}/{total} gambar")
    return predictions

In [ ]:
clip_predictions = predict_clip_topk_batch(ordered_image_paths, LABELS)

result_df = df.copy()
for column in CLIP_COLUMNS:
    result_df[column] = ""

for row_index, file_name in file_series.items():
    if not file_name:
        continue

    for rank, item in enumerate(clip_predictions.get(file_name, []), start=1):
        result_df.at[row_index, f"Clip{rank}_Category"] = item["category"]
        result_df.at[row_index, f"Clip{rank}_Label"] = item["label_id"]

base_columns = list(df.columns)
insert_position = base_columns.index("Catatan") + 1
ordered_columns = base_columns[:insert_position] + CLIP_COLUMNS + base_columns[insert_position:]
result_df = result_df[ordered_columns]

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
result_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"CSV berhasil disimpan: {OUTPUT_CSV}")
print(f"Jumlah baris output: {len(result_df)}")
display(result_df.head())